# neuroSAT — GPU training on Colab

Trains the two PyTorch models on a Colab **GPU** runtime:
* **Part A — Graph-Q-SAT / GAT-Q-SAT** (DQN + GNN, `GQSAT/dqn.py`).
* **Part B — AlphaZeroSAT** (Alpha(Go)Zero + MCTS, `AlphaZeroSAT/train_torch.py`).

Set the runtime to **GPU** first: *Runtime → Change runtime type → GPU*.

**Checkpoints are written directly to Google Drive**, so they survive a Colab
disconnect: if a run is interrupted, just **re-run the training cell** and it
**auto-resumes** from the last checkpoint (GQSAT from `status.yaml`, AlphaZeroSAT
from `az_model.pt`).

## 0. GPU check + clone the repo (everything is on GitHub)

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime → GPU'
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

import os
# mount Drive FIRST: checkpoints are written here so they survive Colab disconnects (and enable resume)
from google.colab import drive
drive.mount('/content/gdrive')
CKPT_DIR = '/content/gdrive/MyDrive/neuroSAT_ckpts'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

# absolute path so re-running cells never nests clones
if not os.path.isdir('/content/neuroSAT'):
    !git clone --recurse-submodules https://github.com/dmeoli/NeuroSAT /content/neuroSAT
%cd /content/neuroSAT
!git -C GQSAT rev-parse --abbrev-ref HEAD; git -C AlphaZeroSAT rev-parse --abbrev-ref HEAD

## 1. Install the modern stack
Colab already ships a CUDA `torch`; we only add the rest (no torch-scatter/sparse).

In [7]:
!pip -q install torch-geometric gymnasium PyYAML tensorboardX scipy
!sudo apt-get -qq install -y swig zlib1g-dev >/dev/null
import sys
PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
!sudo apt-get -qq install -y {PYV}-dev >/dev/null
print('build prereqs ready for', PYV)

build prereqs ready for python3.12


## Part A — Graph-Q-SAT / GAT-Q-SAT (GPU)
### A.1 Build the MiniSat gym extension

In [8]:
import sys, os
PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
NPINC = __import__('numpy').get_include()
%cd /content/neuroSAT/GQSAT/minisat
!make clean >/dev/null 2>&1; true
!make python-wrap PYTHON={PYV} NUMPY_INC="{NPINC}" 2>&1 | tail -6
print('built:', os.path.exists('minisat/gym/_GymSolver.so'))
%cd /content/neuroSAT/GQSAT
import minisat  # registers sat-v0
from minisat.minisat.gym.MiniSATEnv import gym_sat_Env  # noqa: F401  (GQSAT env class; loads the .so)
print('GQSAT env OK')

/content/neuroSAT/GQSAT/minisat
./minisat/core/Dimacs.h:50:9: warning: variable ‘vars’ set but not used [-Wunused-but-set-variable]
   50 |     int vars    = 0;
      |         ^~~~
Linking Shared Library: build/dynamic/lib/libminisat.so.2.1.0
g++ -O2 -fPIC -c minisat/gym/GymSolver_wrap.cxx -o minisat/gym/GymSolver_wrap.o -I. -I/usr/include/python3.12 -I"/usr/local/lib/python3.12/dist-packages/numpy/_core/include"
g++ -shared -o minisat/gym/_GymSolver.so build/dynamic/minisat/core/Solver.o build/dynamic/minisat/simp/SimpSolver.o build/dynamic/minisat/utils/Options.o build/dynamic/minisat/utils/System.o build/dynamic/minisat/gym/GymSolver.o minisat/gym/GymSolver_wrap.o /usr/lib/x86_64-linux-gnu/libz.so
built: True
/content/neuroSAT/GQSAT
GQSAT env OK


### A.2 Train GAT-Q-SAT on graph colouring (GPU)
Drop `--use_attention --heads 3` to train plain Graph-Q-SAT instead. Edit the
data paths / `--batch-updates` as needed. No `--no_cuda` ⇒ it uses the GPU.

In [ ]:
%cd /content/neuroSAT/GQSAT
import os
LOGDIR = f'{CKPT_DIR}/gqsat'          # on Drive -> survives disconnects
os.makedirs(LOGDIR, exist_ok=True)
TRAIN = '../data/graph-coloring/train/flat50-115'
VAL   = '../data/graph-coloring/val/flat50-115'
# AUTO-RESUME: if a status.yaml from a previous (interrupted) run exists on Drive,
# continue from it (restores net + target + optimizer + scheduler + step counter).
RESUME = f'--status_dict_path {LOGDIR}/status.yaml' if os.path.isfile(f'{LOGDIR}/status.yaml') else ''
print('>>> ' + ('RESUMING from ' if RESUME else 'FRESH START -> ') + LOGDIR)
!python3 dqn.py --logdir {LOGDIR} {RESUME} --env-name sat-v0 \
  --train-problems-paths {TRAIN} --eval-problems-paths {VAL} \
  --use_attention --heads 3 \
  --lr 0.00002 --bsize 64 --buffer-size 20000 \
  --eps-init 1.0 --eps-final 0.01 --eps-decay-steps 30000 --gamma 0.99 \
  --batch-updates 50000 --history-len 1 --init-exploration-steps 5000 \
  --step-freq 4 --target-update-freq 10 --loss mse --opt adam \
  --save-freq 500 --grad_clip 1 --grad_clip_norm_type 2 \
  --eval-freq 1000 --eval-time-limit 3600 --core-steps 4 \
  --expert-exploration-prob 0.0 --priority_alpha 0.5 --priority_beta 0.5 \
  --e2v-aggregator sum --n_hidden 1 --hidden_size 64 \
  --decoder_v_out_size 32 --decoder_e_out_size 1 --decoder_g_out_size 1 \
  --encoder_v_out_size 32 --encoder_e_out_size 32 --encoder_g_out_size 32 \
  --core_v_out_size 64 --core_e_out_size 64 --core_g_out_size 32 \
  2>&1 | tee -a {LOGDIR}/gqsat_train.log

In [ ]:
# monitor training (reads the events from the Drive logdir)
%load_ext tensorboard
%tensorboard --logdir /content/gdrive/MyDrive/neuroSAT_ckpts/gqsat

## Part B — AlphaZeroSAT (GPU)
### B.1 Build the MCTSminisat env (GSL-free)

In [ ]:
import os
%cd /content/neuroSAT/AlphaZeroSAT
!PYTHON=python3 bash MCTSminisat/build_so.sh 2>&1 | tail -5
print('built:', os.path.exists('MCTSminisat/minisat/gym/_GymSolver.so'))

### B.2 Self-play + train (GPU)
MCTS self-play runs in the C++ env (CPU); the network trains on the GPU
(`--device auto`). Scale `--cycles` / `--n_batch` / `--n_repeat` for a real run.

In [ ]:
%cd /content/neuroSAT/AlphaZeroSAT
import os
os.makedirs(f'{CKPT_DIR}/alphazero', exist_ok=True)
# train_torch.py loads --save_path if it already exists -> resumes the net weights
# across sessions (so a re-run after a disconnect continues from the saved model).
!python3 train_torch.py \
  --train_path data/uf20-91_train_v0 --device auto \
  --n_batch 16 --n_repeat 10 --resign 400 \
  --cycles 10 --sl_num_steps 1000 --sl_n_batch 32 --lr 1e-3 \
  --save_path {CKPT_DIR}/alphazero/az_model.pt \
  2>&1 | tee -a {CKPT_DIR}/alphazero/az_train.log

## 2. Checkpoints (already on Drive) — resume after a disconnect

Training writes **directly to Google Drive** (`CKPT_DIR`), so checkpoints survive a
Colab disconnect. To continue an interrupted run, just **re-run the training cell**:
* **GQSAT** resumes from `gqsat/status.yaml` (net + optimizer + scheduler + step).
* **AlphaZeroSAT** resumes the net weights from `alphazero/az_model.pt`.

In [ ]:
# inspect what is saved on Drive (checkpoints, status.yaml, tensorboard events, logs)
!ls -lah {CKPT_DIR}/gqsat 2>/dev/null | tail -20
print('---')
!ls -lah {CKPT_DIR}/alphazero 2>/dev/null | tail -10